In [3]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-01-21 14:06:40.001236: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-21 14:06:40.415545: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-21 14:06:40.418071: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-21 14:06:42.178647: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


# Open Data loader & context

In [4]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [5]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [6]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [7]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [8]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [9]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [10]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [11]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [10]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [13]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [14]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [ ]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [15]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [16]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [18]:
ctx.key_games

(array([5557, 6491, 1903,  248, 9446, 4744, 7244, 7104, 1121, 2338, 3864,
        5779, 7137, 2833,  381, 1230, 8071, 1332, 7252, 7008, 3836, 8421,
        2172, 6623, 6689, 8197, 4685, 9364, 5218, 8890, 9380, 1575, 2227,
        3020, 4791, 7807, 1805, 5783, 4066, 9838, 2355, 9200, 4128, 7461,
        5236, 1779, 3046, 5729, 2728, 6605, 9045, 2190, 7592, 1649, 6755,
          89, 6528, 8123, 7599, 2428,  957, 8020, 8998, 5831, 9227,  639,
        4832, 1214,  181, 4184, 4972, 1139, 5304, 6240, 3396, 6563, 3776,
        2262, 4805, 8566, 6095,  102, 9860, 3052, 5068, 3100, 5432, 1326,
        2501,  321,  186, 3516, 5841, 3959, 9276, 8843, 5563, 1502, 5900,
        6003]),
 array([0.34743124, 0.08071299, 0.04348799, 0.04027334, 0.02930198,
        0.0277006 , 0.02327564, 0.01926292, 0.01333025, 0.01233328,
        0.01029388, 0.00868232, 0.00840466, 0.00822039, 0.00820282,
        0.00737349, 0.00651845, 0.00609048, 0.00606077, 0.00505034,
        0.00490937, 0.00482551, 0.00459832, 0.

In [20]:
R = np.load('./R1000train.npz')
R["R"].shape

(1000, 8223)

In [23]:
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=100
)

Best det =  0.0
Current de =  0.0
100 599 300


In [25]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7fb6e624cc50>
np.mean(results), mse, len(results) =  0.025575959933222037 0.0054122903 599
np.mean(results), mse, len(results) =  0.021199999999999997 0.0055082804 300
0.025575959933222037 0.021199999999999997



model =  AnnCur(140423420926096)
np.mean(results), mse, len(results) =  0.6999833055091819 0.0011132385 599
np.mean(results), mse, len(results) =  0.6327333333333333 0.0016203691 300
0.6999833055091819 0.6327333333333333



model =  AnnCur(key_games=np.arange(100) - 140423419826128)
np.mean(results), mse, len(results) =  0.6657929883138565 0.0013816745 599
np.mean(results), mse, len(results) =  0.5838000000000001 0.0020310022 300
0.6657929883138565 0.5838000000000001



model =  AnnCur(random - 140423506349776)
np.mean(results), mse, len(results) =  0.6997829716193656 0.0011298328 599
np.mean(results), mse, len(results) =  0.6371333333333333 0.0016117776 300
0.6997829716193656 0.6371333333333333



model =  AnnCur(K_KMeans - 1404234216

In [27]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7360267111853088 0.00086447876 599
np.mean(results), mse, len(results) =  0.6599666666666667 0.0012965548 300


(0.7360267111853088, 0.6599666666666667)

In [28]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.012199999999999999 tf.Tensor(40.335823, shape=(), dtype=float32) 100
slice = key, score = 0.012199999999999999
np.mean(results), mse, len(results) =  0.01297161936560935 tf.Tensor(39.4795, shape=(), dtype=float32) 599
slice = train, score = 0.01297161936560935
np.mean(results), mse, len(results) =  0.0108 tf.Tensor(39.59065, shape=(), dtype=float32) 300
slice = test, score = 0.0108

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6694 tf.Tensor(53.029842, shape=(), dtype=float32) 100
slice = key, score = 0.6694
np.mean(results), mse, len(results) =  0.700534223706177 tf.Tensor(50.63247, shape=(), dtype=float32) 599
slice = train, score = 0.700534223706177
np.mea


=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.767 tf.Tensor(288.67142, shape=(), dtype=float32) 100
slice = key, score = 0.767
np.mean(results), mse, len(results) =  0.7998163606010016 tf.Tensor(273.83337, shape=(), dtype=float32) 599
slice = train, score = 0.7998163606010016
np.mean(results), mse, len(results) =  0.6234666666666667 tf.Tensor(282.08292, shape=(), dtype=float32) 300
slice = test, score = 0.6234666666666667

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.7722 tf.Tensor(296.29333, shape=(), dtype=float32) 100
slice = key, score = 0.7722
np.mean(results), mse, len(results) =  0.8015358931552587 tf.Tensor(281.067, shape=(), dtype=float32) 599
slice = train, score = 0.8015358931552587
np.mean(results), mse, len(results) =  0.6213333333333334 tf.Tensor(287.613, shape=(), dtype=float32) 300
slice = test, score = 0.6213333333333334

=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.7707999999999999 tf.Tensor(304.1817, shape=(

KeyboardInterrupt: 

In [29]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.011400000000000004 tf.Tensor(45.280064, shape=(), dtype=float32) 100
slice = key, score = 0.011400000000000004
np.mean(results), mse, len(results) =  0.012270450751252087 tf.Tensor(44.569244, shape=(), dtype=float32) 599
slice = train, score = 0.012270450751252087
np.mean(results), mse, len(results) =  0.013933333333333332 tf.Tensor(43.61967, shape=(), dtype=float32) 300
slice = test, score = 0.013933333333333332


KeyboardInterrupt: 

# shuffle

In [34]:
R = np.load('./R1000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=True
)

Best det =  0.0
Current de =  0.0
100 599 300


In [35]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.737295492487479 0.0008616203 599
np.mean(results), mse, len(results) =  0.6681666666666666 0.0013216065 300


(0.737295492487479, 0.6681666666666666)

In [36]:
R = np.load('./R1000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 599 300


In [37]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7360267111853088 0.00086447876 599
np.mean(results), mse, len(results) =  0.6599666666666667 0.0012965548 300


(0.7360267111853088, 0.6599666666666667)

In [38]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.019900000000000004 tf.Tensor(41.79086, shape=(), dtype=float32) 100
slice = key, score = 0.019900000000000004
np.mean(results), mse, len(results) =  0.018914858096828045 tf.Tensor(42.32535, shape=(), dtype=float32) 599
slice = train, score = 0.018914858096828045
np.mean(results), mse, len(results) =  0.015333333333333336 tf.Tensor(41.833496, shape=(), dtype=float32) 300
slice = test, score = 0.015333333333333336

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6730999999999999 tf.Tensor(56.058117, shape=(), dtype=float32) 100
slice = key, score = 0.6730999999999999
np.mean(results), mse, len(results) =  0.6961602671118531 tf.Tensor(53.612713, shape=(), dtype=flo


KeyboardInterrupt



In [39]:
R = np.load('./R2000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1299 600


In [40]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(





model =  <__main__.Popular object at 0x7fb3b6e2ad50>
np.mean(results), mse, len(results) =  0.0131562740569669 0.0052481424 1299
np.mean(results), mse, len(results) =  0.0123 0.0052285064 600
0.0131562740569669 0.0123



model =  AnnCur(140409844430416)
np.mean(results), mse, len(results) =  0.6497613548883756 0.001259694 1299
np.mean(results), mse, len(results) =  0.61785 0.0014691618 600
0.6497613548883756 0.61785



model =  AnnCur(key_games=np.arange(100) - 140409845024464)
np.mean(results), mse, len(results) =  0.6214934565050038 0.0014953915 1299
np.mean(results), mse, len(results) =  0.5832333333333334 0.001780635 600
0.6214934565050038 0.5832333333333334



model =  AnnCur(random - 140409844474704)
np.mean(results), mse, len(results) =  0.6439799846035411 0.0012796524 1299
np.mean(results), mse, len(results) =  0.6204666666666667 0.0014895962 600
0.6439799846035411 0.6204666666666667



model =  AnnCur(K_KMeans - 140409844810960)
np.mean(results), mse, len(results) =  0.6535

In [41]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.687359507313318 0.00095816434 1299
np.mean(results), mse, len(results) =  0.6530333333333335 0.001170445 600


(0.687359507313318, 0.6530333333333335)

In [42]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3566 tf.Tensor(0.006995185094597709, shape=(), dtype=float64) 100
slice = key, score = 0.3566
np.mean(results), mse, len(results) =  0.37841416474210937 tf.Tensor(0.006715046559502405, shape=(), dtype=float64) 1299
slice = train, score = 0.37841416474210937
np.mean(results), mse, len(results) =  0.3667166666666667 tf.Tensor(0.006721360176700356, shape=(), dtype=float64) 600
slice = test, score = 0.3667166666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6606000000000002 tf.Tensor(0.0011121411983254894, shape=(), dtype=float64) 100
slice = key, score = 0.6606000000000002
np.mean(results), mse, len(results) =  0.6864357197844495 tf.Tensor(0.0009623541

np.mean(results), mse, len(results) =  0.6984911470361816 tf.Tensor(0.0008955449946316606, shape=(), dtype=float64) 1299
slice = train, score = 0.6984911470361816
np.mean(results), mse, len(results) =  0.6545500000000001 tf.Tensor(0.0011795267476870834, shape=(), dtype=float64) 600
slice = test, score = 0.6545500000000001

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6778000000000002 tf.Tensor(0.0010225696830068575, shape=(), dtype=float64) 100
slice = key, score = 0.6778000000000002
np.mean(results), mse, len(results) =  0.6987990762124712 tf.Tensor(0.000893962429318988, shape=(), dtype=float64) 1299
slice = train, score = 0.6987990762124712
np.mean(results), mse, len(results) =  0.6552 tf.Tensor(0.0011797502920867303, shape=(), dtype=float64) 600
slice = test, score = 0.6552

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6790999999999998 tf.Tensor(0.0010240072675129777, shape=(), dtype=float64) 100
slice = key, score = 0.6790999999999998
np.mean


=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.6809000000000001 tf.Tensor(0.001001586792664391, shape=(), dtype=float64) 100
slice = key, score = 0.6809000000000001
np.mean(results), mse, len(results) =  0.7022401847575058 tf.Tensor(0.000876066972540026, shape=(), dtype=float64) 1299
slice = train, score = 0.7022401847575058
np.mean(results), mse, len(results) =  0.6544 tf.Tensor(0.0011950224690866545, shape=(), dtype=float64) 600
slice = test, score = 0.6544

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6812 tf.Tensor(0.000999434232135334, shape=(), dtype=float64) 100
slice = key, score = 0.6812
np.mean(results), mse, len(results) =  0.7025019245573517 tf.Tensor(0.0008745641469216709, shape=(), dtype=float64) 1299
slice = train, score = 0.7025019245573517
np.mean(results), mse, len(results) =  0.65445 tf.Tensor(0.0011951560751542348, shape=(), dtype=float64) 600
slice = test, score = 0.65445

=== Iteration 35000 ===
np.mean(results), mse, len(res


=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.6856999999999998 tf.Tensor(0.0009794553685376715, shape=(), dtype=float64) 100
slice = key, score = 0.6856999999999998
np.mean(results), mse, len(results) =  0.7075057736720554 tf.Tensor(0.0008573026397826302, shape=(), dtype=float64) 1299
slice = train, score = 0.7075057736720554
np.mean(results), mse, len(results) =  0.655 tf.Tensor(0.001204198728470459, shape=(), dtype=float64) 600
slice = test, score = 0.655

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.6846 tf.Tensor(0.0009783312692450237, shape=(), dtype=float64) 100
slice = key, score = 0.6846
np.mean(results), mse, len(results) =  0.707590454195535 tf.Tensor(0.0008577708452796023, shape=(), dtype=float64) 1299
slice = train, score = 0.707590454195535
np.mean(results), mse, len(results) =  0.6550166666666667 tf.Tensor(0.0012063401086120566, shape=(), dtype=float64) 600
slice = test, score = 0.6550166666666667

=== Iteration 52000 ===
np.mean(re

np.mean(results), mse, len(results) =  0.6549833333333334 tf.Tensor(0.0012185069754498336, shape=(), dtype=float64) 600
slice = test, score = 0.6549833333333334

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.6898000000000001 tf.Tensor(0.0009619391117057332, shape=(), dtype=float64) 100
slice = key, score = 0.6898000000000001
np.mean(results), mse, len(results) =  0.7114241724403387 tf.Tensor(0.0008430188036816006, shape=(), dtype=float64) 1299
slice = train, score = 0.7114241724403387
np.mean(results), mse, len(results) =  0.6555166666666667 tf.Tensor(0.0012182833115772906, shape=(), dtype=float64) 600
slice = test, score = 0.6555166666666667

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.6901000000000002 tf.Tensor(0.000964411819567814, shape=(), dtype=float64) 100
slice = key, score = 0.6901000000000002
np.mean(results), mse, len(results) =  0.7114472671285604 tf.Tensor(0.0008436858077908951, shape=(), dtype=float64) 1299
slice = train, score = 0.

np.mean(results), mse, len(results) =  0.7137952270977675 tf.Tensor(0.0008350182686561509, shape=(), dtype=float64) 1299
slice = train, score = 0.7137952270977675
np.mean(results), mse, len(results) =  0.6550166666666667 tf.Tensor(0.0012290690590314524, shape=(), dtype=float64) 600
slice = test, score = 0.6550166666666667

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6918000000000001 tf.Tensor(0.0009515427472811162, shape=(), dtype=float64) 100
slice = key, score = 0.6918000000000001
np.mean(results), mse, len(results) =  0.7139568899153195 tf.Tensor(0.0008340069068868251, shape=(), dtype=float64) 1299
slice = train, score = 0.7139568899153195
np.mean(results), mse, len(results) =  0.6544 tf.Tensor(0.0012306968810055243, shape=(), dtype=float64) 600
slice = test, score = 0.6544

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6916 tf.Tensor(0.0009505937251045995, shape=(), dtype=float64) 100
slice = key, score = 0.6916
np.mean(results), mse, len(res

(0.7155658198614319, 0.6544833333333333)

In [43]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3342000000000001 tf.Tensor(0.00812660836522136, shape=(), dtype=float64) 100
slice = key, score = 0.3342000000000001
np.mean(results), mse, len(results) =  0.3460123171670516 tf.Tensor(0.00813912568104136, shape=(), dtype=float64) 1299
slice = train, score = 0.3460123171670516
np.mean(results), mse, len(results) =  0.33418333333333333 tf.Tensor(0.008060184601284569, shape=(), dtype=float64) 600
slice = test, score = 0.33418333333333333

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2625 tf.Tensor(60.13571688826598, shape=(), dtype=float64) 100
slice = key, score = 0.2625
np.mean(results), mse, len(results) =  0.27375673595073136 tf.Tensor(60.16359005293123

np.mean(results), mse, len(results) =  0.5169 tf.Tensor(317.8685754721679, shape=(), dtype=float64) 600
slice = test, score = 0.5169

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5709000000000001 tf.Tensor(422.3722907441085, shape=(), dtype=float64) 100
slice = key, score = 0.5709000000000001
np.mean(results), mse, len(results) =  0.5819014626635874 tf.Tensor(354.66781186924845, shape=(), dtype=float64) 1299
slice = train, score = 0.5819014626635874
np.mean(results), mse, len(results) =  0.5189666666666667 tf.Tensor(340.88742820686406, shape=(), dtype=float64) 600
slice = test, score = 0.5189666666666667

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5778 tf.Tensor(424.50821590358964, shape=(), dtype=float64) 100
slice = key, score = 0.5778
np.mean(results), mse, len(results) =  0.5847036181678213 tf.Tensor(363.193559551612, shape=(), dtype=float64) 1299
slice = train, score = 0.5847036181678213
np.mean(results), mse, len(results) =  0.51968333333

np.mean(results), mse, len(results) =  0.5450166666666667 tf.Tensor(545.1012912887024, shape=(), dtype=float64) 600
slice = test, score = 0.5450166666666667

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6121 tf.Tensor(638.8818181742741, shape=(), dtype=float64) 100
slice = key, score = 0.6121
np.mean(results), mse, len(results) =  0.6183756735950732 tf.Tensor(555.093590680943, shape=(), dtype=float64) 1299
slice = train, score = 0.6183756735950732
np.mean(results), mse, len(results) =  0.5464166666666667 tf.Tensor(538.1771856604648, shape=(), dtype=float64) 600
slice = test, score = 0.5464166666666667

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.6114 tf.Tensor(670.5675026101087, shape=(), dtype=float64) 100
slice = key, score = 0.6114
np.mean(results), mse, len(results) =  0.61846805234796 tf.Tensor(583.7058887051679, shape=(), dtype=float64) 1299
slice = train, score = 0.61846805234796
np.mean(results), mse, len(results) =  0.54655 tf.Tensor(56


=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.626 tf.Tensor(912.8256226443937, shape=(), dtype=float64) 100
slice = key, score = 0.626
np.mean(results), mse, len(results) =  0.6376443418013856 tf.Tensor(793.4649614846069, shape=(), dtype=float64) 1299
slice = train, score = 0.6376443418013856
np.mean(results), mse, len(results) =  0.5648500000000001 tf.Tensor(772.0378450415528, shape=(), dtype=float64) 600
slice = test, score = 0.5648500000000001

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.6275 tf.Tensor(939.7405248730755, shape=(), dtype=float64) 100
slice = key, score = 0.6275
np.mean(results), mse, len(results) =  0.6375288683602772 tf.Tensor(805.3023315772184, shape=(), dtype=float64) 1299
slice = train, score = 0.6375288683602772
np.mean(results), mse, len(results) =  0.5646833333333333 tf.Tensor(785.5762477547873, shape=(), dtype=float64) 600
slice = test, score = 0.5646833333333333

=== Iteration 53000 ===
np.mean(results), mse, len(resu

np.mean(results), mse, len(results) =  0.6496689761354889 tf.Tensor(1021.0783000369535, shape=(), dtype=float64) 1299
slice = train, score = 0.6496689761354889
np.mean(results), mse, len(results) =  0.5745333333333333 tf.Tensor(994.1650788562497, shape=(), dtype=float64) 600
slice = test, score = 0.5745333333333333

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6358000000000001 tf.Tensor(1202.437189527943, shape=(), dtype=float64) 100
slice = key, score = 0.6358000000000001
np.mean(results), mse, len(results) =  0.6510084680523479 tf.Tensor(1060.7080018117979, shape=(), dtype=float64) 1299
slice = train, score = 0.6510084680523479
np.mean(results), mse, len(results) =  0.5754 tf.Tensor(1036.4435615742175, shape=(), dtype=float64) 600
slice = test, score = 0.5754

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6361 tf.Tensor(1220.6197500221524, shape=(), dtype=float64) 100
slice = key, score = 0.6361
np.mean(results), mse, len(results) =  0.651693610

np.mean(results), mse, len(results) =  0.5800166666666667 tf.Tensor(1284.9949234115932, shape=(), dtype=float64) 600
slice = test, score = 0.5800166666666667

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6428 tf.Tensor(1547.9146195984931, shape=(), dtype=float64) 100
slice = key, score = 0.6428
np.mean(results), mse, len(results) =  0.6605311778290993 tf.Tensor(1367.1826616895219, shape=(), dtype=float64) 1299
slice = train, score = 0.6605311778290993
np.mean(results), mse, len(results) =  0.58145 tf.Tensor(1326.3792204005606, shape=(), dtype=float64) 600
slice = test, score = 0.58145

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.6444 tf.Tensor(1529.7461105814132, shape=(), dtype=float64) 100
slice = key, score = 0.6444
np.mean(results), mse, len(results) =  0.6603310238645111 tf.Tensor(1355.053967786398, shape=(), dtype=float64) 1299
slice = train, score = 0.6603310238645111
np.mean(results), mse, len(results) =  0.5809333333333334 tf.Tensor(131

(0.6661200923787529, 0.5848)

In [44]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3342 tf.Tensor(0.008297997500201588, shape=(), dtype=float64) 100
slice = key, score = 0.3342
np.mean(results), mse, len(results) =  0.35704387990762126 tf.Tensor(0.007691356497170854, shape=(), dtype=float64) 1299
slice = train, score = 0.35704387990762126
np.mean(results), mse, len(results) =  0.34868333333333335 tf.Tensor(0.007626336463157864, shape=(), dtype=float64) 600
slice = test, score = 0.34868333333333335

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2422 tf.Tensor(57.00378808136002, shape=(), dtype=float64) 100
slice = key, score = 0.2422
np.mean(results), mse, len(results) =  0.26941493456505006 tf.Tensor(55.56765585783994, shape=(), dtype=fl

np.mean(results), mse, len(results) =  0.5753348729792148 tf.Tensor(359.94595988084217, shape=(), dtype=float64) 1299
slice = train, score = 0.5753348729792148
np.mean(results), mse, len(results) =  0.5091833333333333 tf.Tensor(326.4790746392788, shape=(), dtype=float64) 600
slice = test, score = 0.5091833333333333

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5128 tf.Tensor(394.58519492750736, shape=(), dtype=float64) 100
slice = key, score = 0.5128
np.mean(results), mse, len(results) =  0.5792070823710546 tf.Tensor(361.3697791567608, shape=(), dtype=float64) 1299
slice = train, score = 0.5792070823710546
np.mean(results), mse, len(results) =  0.5114000000000001 tf.Tensor(325.59866828275915, shape=(), dtype=float64) 600
slice = test, score = 0.5114000000000001

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5167 tf.Tensor(412.0174965055812, shape=(), dtype=float64) 100
slice = key, score = 0.5167
np.mean(results), mse, len(results) =  0.5822940723


=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5493 tf.Tensor(612.0432468930535, shape=(), dtype=float64) 100
slice = key, score = 0.5493
np.mean(results), mse, len(results) =  0.619153194765204 tf.Tensor(551.6594966508841, shape=(), dtype=float64) 1299
slice = train, score = 0.619153194765204
np.mean(results), mse, len(results) =  0.5454833333333333 tf.Tensor(512.5565170680536, shape=(), dtype=float64) 600
slice = test, score = 0.5454833333333333

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5458999999999999 tf.Tensor(606.1851649637375, shape=(), dtype=float64) 100
slice = key, score = 0.5458999999999999
np.mean(results), mse, len(results) =  0.6201616628175519 tf.Tensor(549.3799744589634, shape=(), dtype=float64) 1299
slice = train, score = 0.6201616628175519
np.mean(results), mse, len(results) =  0.5464333333333333 tf.Tensor(511.70981080495505, shape=(), dtype=float64) 600
slice = test, score = 0.5464333333333333

=== Iteration 36000 ===
np.mea

np.mean(results), mse, len(results) =  0.6381678214010778 tf.Tensor(706.4889449488097, shape=(), dtype=float64) 1299
slice = train, score = 0.6381678214010778
np.mean(results), mse, len(results) =  0.5626166666666667 tf.Tensor(671.3241728849277, shape=(), dtype=float64) 600
slice = test, score = 0.5626166666666667

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5623 tf.Tensor(780.3608888571165, shape=(), dtype=float64) 100
slice = key, score = 0.5623
np.mean(results), mse, len(results) =  0.6383987682832948 tf.Tensor(712.9613496225091, shape=(), dtype=float64) 1299
slice = train, score = 0.6383987682832948
np.mean(results), mse, len(results) =  0.5618833333333333 tf.Tensor(681.5567475055398, shape=(), dtype=float64) 600
slice = test, score = 0.5618833333333333

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.563 tf.Tensor(803.5309206606188, shape=(), dtype=float64) 100
slice = key, score = 0.563
np.mean(results), mse, len(results) =  0.639245573518090

np.mean(results), mse, len(results) =  0.5727166666666667 tf.Tensor(868.3089323370391, shape=(), dtype=float64) 600
slice = test, score = 0.5727166666666667

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5725000000000001 tf.Tensor(1030.5539001441236, shape=(), dtype=float64) 100
slice = key, score = 0.5725000000000001
np.mean(results), mse, len(results) =  0.6527405696689762 tf.Tensor(922.5814767142924, shape=(), dtype=float64) 1299
slice = train, score = 0.6527405696689762
np.mean(results), mse, len(results) =  0.5730000000000001 tf.Tensor(877.0432301029147, shape=(), dtype=float64) 600
slice = test, score = 0.5730000000000001

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.5719 tf.Tensor(1053.8090374938347, shape=(), dtype=float64) 100
slice = key, score = 0.5719
np.mean(results), mse, len(results) =  0.6524942263279445 tf.Tensor(933.9735229356652, shape=(), dtype=float64) 1299
slice = train, score = 0.6524942263279445
np.mean(results), mse, len(r


=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.5818 tf.Tensor(1230.2911315726462, shape=(), dtype=float64) 100
slice = key, score = 0.5818
np.mean(results), mse, len(results) =  0.6653117782909931 tf.Tensor(1120.5745411390492, shape=(), dtype=float64) 1299
slice = train, score = 0.6653117782909931
np.mean(results), mse, len(results) =  0.5824833333333334 tf.Tensor(1061.7782362053788, shape=(), dtype=float64) 600
slice = test, score = 0.5824833333333334

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.5797 tf.Tensor(1260.7177305889202, shape=(), dtype=float64) 100
slice = key, score = 0.5797
np.mean(results), mse, len(results) =  0.6653810623556582 tf.Tensor(1132.4865571119271, shape=(), dtype=float64) 1299
slice = train, score = 0.6653810623556582
np.mean(results), mse, len(results) =  0.5827666666666667 tf.Tensor(1074.1518916108253, shape=(), dtype=float64) 600
slice = test, score = 0.5827666666666667

=== Iteration 88000 ===
np.mean(results), mse, 

(0.6716474210931486, 0.5864166666666667)

In [45]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.32959999999999995 tf.Tensor(0.008082917797811014, shape=(), dtype=float64) 100
slice = key, score = 0.32959999999999995
np.mean(results), mse, len(results) =  0.34551963048498846 tf.Tensor(0.0077616020443872, shape=(), dtype=float64) 1299
slice = train, score = 0.34551963048498846
np.mean(results), mse, len(results) =  0.33845 tf.Tensor(0.007700447712871076, shape=(), dtype=float64) 600
slice = test, score = 0.33845

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6574 tf.Tensor(0.0011422899621397783, shape=(), dtype=float64) 100
slice = key, score = 0.6574
np.mean(results), mse, len(results) =  0.6871747498075443 tf.Tensor(0.000962104919103781, shape=(), dt

np.mean(results), mse, len(results) =  0.6987143956889915 tf.Tensor(0.0008951755874591951, shape=(), dtype=float64) 1299
slice = train, score = 0.6987143956889915
np.mean(results), mse, len(results) =  0.6518999999999999 tf.Tensor(0.0011894847874242187, shape=(), dtype=float64) 600
slice = test, score = 0.6518999999999999

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6517000000000001 tf.Tensor(0.0011759019750245644, shape=(), dtype=float64) 100
slice = key, score = 0.6517000000000001
np.mean(results), mse, len(results) =  0.6998922247882987 tf.Tensor(0.0008915228349089248, shape=(), dtype=float64) 1299
slice = train, score = 0.6998922247882987
np.mean(results), mse, len(results) =  0.6524833333333333 tf.Tensor(0.0011899700550185215, shape=(), dtype=float64) 600
slice = test, score = 0.6524833333333333

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6528 tf.Tensor(0.0011743966479080307, shape=(), dtype=float64) 100
slice = key, score = 0.6528
np.mea


=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.6511 tf.Tensor(0.0011958674907363758, shape=(), dtype=float64) 100
slice = key, score = 0.6511
np.mean(results), mse, len(results) =  0.7034103156274057 tf.Tensor(0.0008705157279966964, shape=(), dtype=float64) 1299
slice = train, score = 0.7034103156274057
np.mean(results), mse, len(results) =  0.6523666666666667 tf.Tensor(0.00120481082386952, shape=(), dtype=float64) 600
slice = test, score = 0.6523666666666667

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6523 tf.Tensor(0.00119426101044166, shape=(), dtype=float64) 100
slice = key, score = 0.6523
np.mean(results), mse, len(results) =  0.704464973056197 tf.Tensor(0.000869277443090764, shape=(), dtype=float64) 1299
slice = train, score = 0.704464973056197
np.mean(results), mse, len(results) =  0.6527499999999999 tf.Tensor(0.0012047414186330335, shape=(), dtype=float64) 600
slice = test, score = 0.6527499999999999

=== Iteration 35000 ===
np.mean(resu


=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.6526000000000001 tf.Tensor(0.001207262313295721, shape=(), dtype=float64) 100
slice = key, score = 0.6526000000000001
np.mean(results), mse, len(results) =  0.7087836797536566 tf.Tensor(0.0008519086088079413, shape=(), dtype=float64) 1299
slice = train, score = 0.7087836797536566
np.mean(results), mse, len(results) =  0.6532166666666667 tf.Tensor(0.001216949025188883, shape=(), dtype=float64) 600
slice = test, score = 0.6532166666666667

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.6505 tf.Tensor(0.0012102819722292454, shape=(), dtype=float64) 100
slice = key, score = 0.6505
np.mean(results), mse, len(results) =  0.7088760585065436 tf.Tensor(0.0008504581355051086, shape=(), dtype=float64) 1299
slice = train, score = 0.7088760585065436
np.mean(results), mse, len(results) =  0.6528833333333334 tf.Tensor(0.001218142287557392, shape=(), dtype=float64) 600
slice = test, score = 0.6528833333333334

=== Iter

np.mean(results), mse, len(results) =  0.6524 tf.Tensor(0.001229042158493553, shape=(), dtype=float64) 600
slice = test, score = 0.6524

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.6519999999999999 tf.Tensor(0.0012247070981051074, shape=(), dtype=float64) 100
slice = key, score = 0.6519999999999999
np.mean(results), mse, len(results) =  0.7122940723633564 tf.Tensor(0.0008395960962865434, shape=(), dtype=float64) 1299
slice = train, score = 0.7122940723633564
np.mean(results), mse, len(results) =  0.6520833333333333 tf.Tensor(0.0012328068437780434, shape=(), dtype=float64) 600
slice = test, score = 0.6520833333333333

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6509999999999999 tf.Tensor(0.0012275400993098952, shape=(), dtype=float64) 100
slice = key, score = 0.6509999999999999
np.mean(results), mse, len(results) =  0.7123479599692071 tf.Tensor(0.0008390226773119315, shape=(), dtype=float64) 1299
slice = train, score = 0.7123479599692071
np.mean

np.mean(results), mse, len(results) =  0.6516666666666666 tf.Tensor(0.0012437734893002753, shape=(), dtype=float64) 600
slice = test, score = 0.6516666666666666

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6503999999999999 tf.Tensor(0.001246057752425167, shape=(), dtype=float64) 100
slice = key, score = 0.6503999999999999
np.mean(results), mse, len(results) =  0.7144572748267898 tf.Tensor(0.0008329323432614703, shape=(), dtype=float64) 1299
slice = train, score = 0.7144572748267898
np.mean(results), mse, len(results) =  0.6516500000000001 tf.Tensor(0.0012484274897675302, shape=(), dtype=float64) 600
slice = test, score = 0.6516500000000001

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6522 tf.Tensor(0.0012425769605109685, shape=(), dtype=float64) 100
slice = key, score = 0.6522
np.mean(results), mse, len(results) =  0.714657428791378 tf.Tensor(0.0008299782384428648, shape=(), dtype=float64) 1299
slice = train, score = 0.714657428791378
np.mean(r

(0.7161893764434181, 0.6513666666666666)

In [46]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3051 tf.Tensor(0.009262073986447673, shape=(), dtype=float64) 100
slice = key, score = 0.3051
np.mean(results), mse, len(results) =  0.34113163972286376 tf.Tensor(0.008592258682274378, shape=(), dtype=float64) 1299
slice = train, score = 0.34113163972286376
np.mean(results), mse, len(results) =  0.33048333333333335 tf.Tensor(0.008564824782153313, shape=(), dtype=float64) 600
slice = test, score = 0.33048333333333335

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.6576000000000002 tf.Tensor(0.0011449236396029678, shape=(), dtype=float64) 100
slice = key, score = 0.6576000000000002
np.mean(results), mse, len(results) =  0.6862586605080832 tf.Tensor(0.00096465

np.mean(results), mse, len(results) =  0.6973979984603542 tf.Tensor(0.0009072599659665323, shape=(), dtype=float64) 1299
slice = train, score = 0.6973979984603542
np.mean(results), mse, len(results) =  0.6531666666666667 tf.Tensor(0.0011872477362343163, shape=(), dtype=float64) 600
slice = test, score = 0.6531666666666667

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6552 tf.Tensor(0.0011699926632483414, shape=(), dtype=float64) 100
slice = key, score = 0.6552
np.mean(results), mse, len(results) =  0.6968052347959969 tf.Tensor(0.0009041860359258145, shape=(), dtype=float64) 1299
slice = train, score = 0.6968052347959969
np.mean(results), mse, len(results) =  0.6524 tf.Tensor(0.0011865929735240615, shape=(), dtype=float64) 600
slice = test, score = 0.6524

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.6569000000000002 tf.Tensor(0.001171071019390094, shape=(), dtype=float64) 100
slice = key, score = 0.6569000000000002
np.mean(results), mse, len(resu

np.mean(results), mse, len(results) =  0.7030715935334872 tf.Tensor(0.000880203759960482, shape=(), dtype=float64) 1299
slice = train, score = 0.7030715935334872
np.mean(results), mse, len(results) =  0.6540333333333334 tf.Tensor(0.0011971209233756289, shape=(), dtype=float64) 600
slice = test, score = 0.6540333333333334

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.6555000000000001 tf.Tensor(0.0011874889033263258, shape=(), dtype=float64) 100
slice = key, score = 0.6555000000000001
np.mean(results), mse, len(results) =  0.704033872209392 tf.Tensor(0.000877924989290989, shape=(), dtype=float64) 1299
slice = train, score = 0.704033872209392
np.mean(results), mse, len(results) =  0.65375 tf.Tensor(0.001199713784400069, shape=(), dtype=float64) 600
slice = test, score = 0.65375

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.6548999999999999 tf.Tensor(0.0011934148280070406, shape=(), dtype=float64) 100
slice = key, score = 0.6548999999999999
np.mean(r

np.mean(results), mse, len(results) =  0.7081524249422634 tf.Tensor(0.0008578044072071329, shape=(), dtype=float64) 1299
slice = train, score = 0.7081524249422634
np.mean(results), mse, len(results) =  0.654 tf.Tensor(0.0012099622396409132, shape=(), dtype=float64) 600
slice = test, score = 0.654

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.6522 tf.Tensor(0.0012141780879812417, shape=(), dtype=float64) 100
slice = key, score = 0.6522
np.mean(results), mse, len(results) =  0.7081062355658199 tf.Tensor(0.0008602222610544887, shape=(), dtype=float64) 1299
slice = train, score = 0.7081062355658199
np.mean(results), mse, len(results) =  0.6525666666666667 tf.Tensor(0.0012132916493574734, shape=(), dtype=float64) 600
slice = test, score = 0.6525666666666667

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.6536000000000002 tf.Tensor(0.0012120281728488704, shape=(), dtype=float64) 100
slice = key, score = 0.6536000000000002
np.mean(results), mse, len(resul

np.mean(results), mse, len(results) =  0.7116320246343341 tf.Tensor(0.0008452959563484814, shape=(), dtype=float64) 1299
slice = train, score = 0.7116320246343341
np.mean(results), mse, len(results) =  0.6533333333333333 tf.Tensor(0.0012231435361437046, shape=(), dtype=float64) 600
slice = test, score = 0.6533333333333333

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6532000000000001 tf.Tensor(0.0012386321207258529, shape=(), dtype=float64) 100
slice = key, score = 0.6532000000000001
np.mean(results), mse, len(results) =  0.7120092378752887 tf.Tensor(0.0008476601330973018, shape=(), dtype=float64) 1299
slice = train, score = 0.7120092378752887
np.mean(results), mse, len(results) =  0.6532166666666667 tf.Tensor(0.0012270206781289184, shape=(), dtype=float64) 600
slice = test, score = 0.6532166666666667

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.6535 tf.Tensor(0.0012374713315031316, shape=(), dtype=float64) 100
slice = key, score = 0.6535
np.mea


=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6533 tf.Tensor(0.00125240385536979, shape=(), dtype=float64) 100
slice = key, score = 0.6533
np.mean(results), mse, len(results) =  0.7142956120092379 tf.Tensor(0.0008374109726557612, shape=(), dtype=float64) 1299
slice = train, score = 0.7142956120092379
np.mean(results), mse, len(results) =  0.6523333333333334 tf.Tensor(0.0012376217072447566, shape=(), dtype=float64) 600
slice = test, score = 0.6523333333333334

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.6517000000000001 tf.Tensor(0.0012520590951429064, shape=(), dtype=float64) 100
slice = key, score = 0.6517000000000001
np.mean(results), mse, len(results) =  0.7132871439568899 tf.Tensor(0.0008385184206488179, shape=(), dtype=float64) 1299
slice = train, score = 0.7132871439568899
np.mean(results), mse, len(results) =  0.6523333333333333 tf.Tensor(0.0012365607226378084, shape=(), dtype=float64) 600
slice = test, score = 0.6523333333333333

=== Ite

(0.714203233256351, 0.6515000000000001)

In [47]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.34409999999999996 tf.Tensor(0.008172939178262067, shape=(), dtype=float64) 100
slice = key, score = 0.34409999999999996
np.mean(results), mse, len(results) =  0.35885296381832177 tf.Tensor(0.00792730245621984, shape=(), dtype=float64) 1299
slice = train, score = 0.35885296381832177
np.mean(results), mse, len(results) =  0.34626666666666667 tf.Tensor(0.0078080144604552575, shape=(), dtype=float64) 600
slice = test, score = 0.34626666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2305 tf.Tensor(48.52910324848045, shape=(), dtype=float64) 100
slice = key, score = 0.2305
np.mean(results), mse, len(results) =  0.261824480369515 tf.Tensor(50.3829507811

np.mean(results), mse, len(results) =  0.5762047729022325 tf.Tensor(390.0509747488373, shape=(), dtype=float64) 1299
slice = train, score = 0.5762047729022325
np.mean(results), mse, len(results) =  0.5091833333333333 tf.Tensor(362.93944413903597, shape=(), dtype=float64) 600
slice = test, score = 0.5091833333333333

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.518 tf.Tensor(437.17551553692726, shape=(), dtype=float64) 100
slice = key, score = 0.518
np.mean(results), mse, len(results) =  0.5812163202463433 tf.Tensor(399.15790691180086, shape=(), dtype=float64) 1299
slice = train, score = 0.5812163202463433
np.mean(results), mse, len(results) =  0.51295 tf.Tensor(372.64091285279534, shape=(), dtype=float64) 600
slice = test, score = 0.51295

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5217 tf.Tensor(453.9429062521212, shape=(), dtype=float64) 100
slice = key, score = 0.5217
np.mean(results), mse, len(results) =  0.5853579676674365 tf.Tensor(416.65

np.mean(results), mse, len(results) =  0.5458000000000001 tf.Tensor(530.2351152558987, shape=(), dtype=float64) 600
slice = test, score = 0.5458000000000001

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5545 tf.Tensor(626.8319311971471, shape=(), dtype=float64) 100
slice = key, score = 0.5545
np.mean(results), mse, len(results) =  0.6244495765973825 tf.Tensor(570.9455233002061, shape=(), dtype=float64) 1299
slice = train, score = 0.6244495765973825
np.mean(results), mse, len(results) =  0.5465833333333333 tf.Tensor(531.5886415565676, shape=(), dtype=float64) 600
slice = test, score = 0.5465833333333333

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5566 tf.Tensor(615.4339080965519, shape=(), dtype=float64) 100
slice = key, score = 0.5566
np.mean(results), mse, len(results) =  0.6249268668206313 tf.Tensor(567.1520157510347, shape=(), dtype=float64) 1299
slice = train, score = 0.6249268668206313
np.mean(results), mse, len(results) =  0.5493833333333

np.mean(results), mse, len(results) =  0.5651 tf.Tensor(668.3255277715607, shape=(), dtype=float64) 600
slice = test, score = 0.5651

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.5678 tf.Tensor(781.8840228426566, shape=(), dtype=float64) 100
slice = key, score = 0.5678
np.mean(results), mse, len(results) =  0.6451193225558122 tf.Tensor(723.1479382872236, shape=(), dtype=float64) 1299
slice = train, score = 0.6451193225558122
np.mean(results), mse, len(results) =  0.56515 tf.Tensor(673.8629243195116, shape=(), dtype=float64) 600
slice = test, score = 0.56515

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.569 tf.Tensor(780.0716856935474, shape=(), dtype=float64) 100
slice = key, score = 0.569
np.mean(results), mse, len(results) =  0.6465819861431872 tf.Tensor(731.5410081855434, shape=(), dtype=float64) 1299
slice = train, score = 0.6465819861431872
np.mean(results), mse, len(results) =  0.5669166666666666 tf.Tensor(681.4520887495103, shape=(), dtype

KeyboardInterrupt: 

In [15]:
R = np.load('./R3000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [16]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40f5c35ed0>
np.mean(results), mse, len(results) =  0.00812406203101551 0.005156699 1999
np.mean(results), mse, len(results) =  0.009422222222222222 0.0051572192 900
0.00812406203101551 0.009422222222222222



model =  AnnCur(139917042483792)
np.mean(results), mse, len(results) =  0.633711855927964 0.001257913 1999
np.mean(results), mse, len(results) =  0.6069222222222223 0.0013856486 900
0.633711855927964 0.6069222222222223



model =  AnnCur(key_games=np.arange(100) - 139916978251152)
np.mean(results), mse, len(results) =  0.59927963981991 0.0015316498 1999
np.mean(results), mse, len(results) =  0.5688777777777778 0.0016877775 900
0.59927963981991 0.5688777777777778



model =  AnnCur(random - 139916978078864)
np.mean(results), mse, len(results) =  0.6311455727863933 0.0012744617 1999
np.mean(results), mse, len(results) =  0.6039888888888888 0.0014088538 900
0.6311455727863933 0.6039888888888888



model =  AnnCur(K_KMeans - 139919554740176)

In [17]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_750529/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6648624312156077 0.000989191 1999
np.mean(results), mse, len(results) =  0.6390555555555555 0.0011131779 900


(0.6648624312156077, 0.6390555555555555)

In [19]:
ctx.set_kmeans_games_as_key(all_from_labels=False)

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6281640820410204 0.0012411552 1999
np.mean(results), mse, len(results) =  0.6028111111111111 0.001370198 900


(0.6281640820410204, 0.6028111111111111)

In [90]:
R = np.load('./R3000test.npz')
R_ = R["R"].astype("float64") 
ctx = EvalContextRelevs(
    (R_ - R_.mean()) / R_.std(),
    det_attempts=0,
    shuffle=False
)

Best det =  5.0836383112723515e+59
Current de =  5.0836383112723515e+59
100 1999 900


In [91]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7fb1579a5650>
np.mean(results), mse, len(results) =  0.009444722361180592 0.9567697654965333 1999
np.mean(results), mse, len(results) =  0.009166666666666667 0.971850685741731 900
0.009444722361180592 0.009166666666666667



model =  AnnCur(140394774089168)
np.mean(results), mse, len(results) =  0.6309604802401201 0.2417310715297432 1999
np.mean(results), mse, len(results) =  0.6147777777777778 0.26450940184444943 900
0.6309604802401201 0.6147777777777778



model =  AnnCur(key_games=np.arange(100) - 140409843271824)
np.mean(results), mse, len(results) =  0.5956728364182091 0.2882643239306443 1999
np.mean(results), mse, len(results) =  0.5721444444444443 0.31852632814725745 900
0.5956728364182091 0.5721444444444443



model =  AnnCur(random - 140406345444496)
np.mean(results), mse, len(results) =  0.633896948474237 0.23576882801676524 1999
np.mean(results), mse, len(results) =  0.613588888888889 0.2624383474778105 900
0.633896948474237 0.613588

In [ ]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

In [50]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6689094547273637 0.000980972 1999
np.mean(results), mse, len(results) =  0.6458777777777778 0.0011021171 900


(0.6689094547273637, 0.6458777777777778)

In [53]:
ctx.set_kmeans_games_as_key()

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6234267133566783 0.001265143 1999
np.mean(results), mse, len(results) =  0.6084555555555554 0.0013879173 900


(0.6234267133566783, 0.6084555555555554)

In [52]:
dir(ctx)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'games_count',
 'get_kmeans_games',
 'get_relevs',
 'get_requests',
 'get_top_games',
 'key_games',
 'key_relevs',
 'key_size',
 'relevs',
 'reqs_count',
 'set_kmeans_games_as_key',
 'set_top_games_as_key',
 'slices',
 'test_relevs',
 'train_relevs',
 'train_split',
 'try_det_attempts']

In [54]:
R = np.load('./R3000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=True
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [55]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6721460730365183 0.0009835806 1999
np.mean(results), mse, len(results) =  0.6366222222222223 0.0011193526 900


(0.6721460730365183, 0.6366222222222223)

In [56]:
ctx.set_kmeans_games_as_key()

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6298099049524764 0.0012462635 1999
np.mean(results), mse, len(results) =  0.5925555555555555 0.0013914837 900


(0.6298099049524764, 0.5925555555555555)

In [87]:
kr = ctx.relevs[:100, ctx.key_games].astype("float64")

In [85]:
kr = (kr - kr.mean()) 

In [88]:
np.linalg.det(kr)

8.738492240116713e-53

In [83]:
kr.dtype

dtype('float32')

In [20]:
R = np.load('./R3000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [21]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40e95d2e90>
np.mean(results), mse, len(results) =  0.009444722361180592 0.005165653 1999
np.mean(results), mse, len(results) =  0.009166666666666667 0.0052470677 900
0.009444722361180592 0.009166666666666667



model =  AnnCur(139916770041104)
np.mean(results), mse, len(results) =  0.6352426213106552 0.0012897311 1999
np.mean(results), mse, len(results) =  0.6164111111111111 0.0014229873 900
0.6352426213106552 0.6164111111111111



model =  AnnCur(key_games=np.arange(100) - 139916769195984)
np.mean(results), mse, len(results) =  0.5953476738369184 0.0015576119 1999
np.mean(results), mse, len(results) =  0.5720444444444445 0.0017203941 900
0.5953476738369184 0.5720444444444445



model =  AnnCur(random - 139916769190096)
np.mean(results), mse, len(results) =  0.633671835917959 0.0012710841 1999
np.mean(results), mse, len(results) =  0.6130444444444445 0.0014154438 900
0.633671835917959 0.6130444444444445



model =  AnnCur(K_KMeans - 139916770

In [22]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_750529/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6689094547273637 0.000980972 1999
np.mean(results), mse, len(results) =  0.6458777777777778 0.0011021171 900


(0.6689094547273637, 0.6458777777777778)

## R5000test

In [17]:
R = np.load('./R5000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 3399 1500


In [24]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40f5c4af90>
np.mean(results), mse, len(results) =  0.005445719329214474 0.005149558 3399
np.mean(results), mse, len(results) =  0.005253333333333334 0.0051724273 1500
0.005445719329214474 0.005253333333333334



model =  AnnCur(139916976503504)
np.mean(results), mse, len(results) =  0.6149132097675788 0.0013162897 3399
np.mean(results), mse, len(results) =  0.5955066666666666 0.0013957007 1500
0.6149132097675788 0.5955066666666666



model =  AnnCur(key_games=np.arange(100) - 139916976500752)
np.mean(results), mse, len(results) =  0.5783995292733157 0.0015776153 3399
np.mean(results), mse, len(results) =  0.5589733333333333 0.0016660148 1500
0.5783995292733157 0.5589733333333333



model =  AnnCur(random - 139916770160016)
np.mean(results), mse, len(results) =  0.6142071197411003 0.0012941539 3399
np.mean(results), mse, len(results) =  0.59584 0.0013727732 1500
0.6142071197411003 0.59584



model =  AnnCur(K_KMeans - 139916770086288)
np.mean(

In [18]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_602384/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6448455428067078 0.0010061534 3399
np.mean(results), mse, len(results) =  0.6315666666666667 0.0010672992 1500


(0.6448455428067078, 0.6315666666666667)

In [ ]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (3399, 100)
self.embed_games.shape =  (40718, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 40718)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (3399, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.004699999999999999 tf.Tensor(39.029186, shape=(), dtype=float32) 100
slice = key, score = 0.004699999999999999
np.mean(results), mse, len(results) =  0.002362459546925567 tf.Tensor(37.687843, shape=(), dtype=float32) 3399
slice = train, score = 0.002362459546925567
np.mean(results), mse, len(results) =  0.002426666666666667 tf.Tensor(37.627937, shape=(), dtype=float32) 1500
slice = test, score = 0.002426666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5627 tf.Tensor(48.584892, shape=(), dtype=float32) 100
slice = key, score = 0.5627
np.mean(results), mse, len(results) =  0.6158399529273316 tf.Tensor(47.981865, shape=(), dtype=float32) 3399
slice

np.mean(results), mse, len(results) =  0.64824 tf.Tensor(341.99518, shape=(), dtype=float32) 1500
slice = test, score = 0.64824

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6203000000000001 tf.Tensor(386.49255, shape=(), dtype=float32) 100
slice = key, score = 0.6203000000000001
np.mean(results), mse, len(results) =  0.6814121800529569 tf.Tensor(357.30286, shape=(), dtype=float32) 3399
slice = train, score = 0.6814121800529569
np.mean(results), mse, len(results) =  0.6498 tf.Tensor(355.1875, shape=(), dtype=float32) 1500
slice = test, score = 0.6498

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.6187 tf.Tensor(395.77533, shape=(), dtype=float32) 100
slice = key, score = 0.6187
np.mean(results), mse, len(results) =  0.6811768167107972 tf.Tensor(368.38965, shape=(), dtype=float32) 3399
slice = train, score = 0.6811768167107972
np.mean(results), mse, len(results) =  0.6487466666666667 tf.Tensor(364.85995, shape=(), dtype=float32) 1500
slice = test, 

np.mean(results), mse, len(results) =  0.6885319211532803 tf.Tensor(487.84946, shape=(), dtype=float32) 3399
slice = train, score = 0.6885319211532803
np.mean(results), mse, len(results) =  0.6519133333333332 tf.Tensor(487.10318, shape=(), dtype=float32) 1500
slice = test, score = 0.6519133333333332

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.6234 tf.Tensor(516.5979, shape=(), dtype=float32) 100
slice = key, score = 0.6234
np.mean(results), mse, len(results) =  0.6887878787878788 tf.Tensor(496.27496, shape=(), dtype=float32) 3399
slice = train, score = 0.6887878787878788
np.mean(results), mse, len(results) =  0.6514533333333333 tf.Tensor(494.8931, shape=(), dtype=float32) 1500
slice = test, score = 0.6514533333333333

=== Iteration 38000 ===
np.mean(results), mse, len(results) =  0.6255 tf.Tensor(523.55273, shape=(), dtype=float32) 100
slice = key, score = 0.6255
np.mean(results), mse, len(results) =  0.689985289791115 tf.Tensor(499.51956, shape=(), dtype=float32)

np.mean(results), mse, len(results) =  0.6529533333333333 tf.Tensor(549.6328, shape=(), dtype=float32) 1500
slice = test, score = 0.6529533333333333

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.6248 tf.Tensor(563.38446, shape=(), dtype=float32) 100
slice = key, score = 0.6248
np.mean(results), mse, len(results) =  0.6939658723153869 tf.Tensor(549.22705, shape=(), dtype=float32) 3399
slice = train, score = 0.6939658723153869
np.mean(results), mse, len(results) =  0.6525466666666666 tf.Tensor(551.5697, shape=(), dtype=float32) 1500
slice = test, score = 0.6525466666666666

=== Iteration 56000 ===
np.mean(results), mse, len(results) =  0.6216 tf.Tensor(566.75055, shape=(), dtype=float32) 100
slice = key, score = 0.6216
np.mean(results), mse, len(results) =  0.6948278905560459 tf.Tensor(552.5234, shape=(), dtype=float32) 3399
slice = train, score = 0.6948278905560459
np.mean(results), mse, len(results) =  0.6531133333333333 tf.Tensor(556.09485, shape=(), dtype=float32)